In [42]:
import pandas as pd
import os
import csv
import re
import ftfy
import unicodedata
from collections import Counter
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel, Whitespace
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.trainers import BpeTrainer
from tokenizers.processors import TemplateProcessing
from sklearn.model_selection import train_test_split


In [2]:
data = pd.read_csv(r'..\data\processed\final_combined_jokes.csv')
data.head()

,stable_id,joke,topic
0,0000066daaacde54c609a69be4f00c3336480137,Whoever said imitation is the sincerest form o...,"imitation, flattery, minute"
1,00000a79878b3729adc0e47a34dbf5d5484214c0,"You're so fat, they oughta call your dick gary...","oldman, dick, gary"
2,000054bc76448f9b302e6266a72324ecb81d4083,I couldn't sleep last night. I was wondering w...,"night, sun"
3,0000b76ab05f5d5cf9952d109c5ce7b143ae016f,What's 11 & 2? The Cowboys,cowboys
4,00015c7571451cc5eb99c24dd7bc46a5ce46b9a1,I just got done apologizing to my barbershop q...,"barbershop, quartet, bucket"


In [3]:
def clean_jokes(joke):
    """Remove noise and normalize text for tokenization."""
    # Fix encoding
    joke = ftfy.fix_text(str(joke))
    joke = unicodedata.normalize("NFKC", joke)
    
    # Lowercase
    joke = joke.lower()
    
    # Remove URLs, subreddit refs, mentions, hashtags
    joke = re.sub(r'https?://\S+|www\.\S+', '', joke)
    joke = re.sub(r'/r/\S+', '', joke)
    joke = re.sub(r'@\w+|#\w+', '', joke)
    
    # Keep letters, numbers, and common punctuation
    joke = re.sub(r"[^a-z0-9\s.,!?;:'\"()\-\[\]""''…]", " ", joke)
    
    joke = re.sub(r'\.(?:\s*\.)+', '<ELLIPSIS>', joke)
    
    # Reduce repeated punctuation
    joke = re.sub(r'([!?])\1+', r'\1', joke)
         
    # Normalize whitespace
    joke = ' '.join(joke.split())
    
    # Protect contractions
    joke = re.sub(r"(\w)'(\w)", r"\1<APOST>\2", joke)
    
    # Add spacing around punctuation
    joke = re.sub(r"([.!?,;:(){}\[\]\"""''])", r" \1 ", joke)
    
    # Collapse extra spaces
    joke = re.sub(r"\s{2,}", " ", joke).strip()
    
    # Restore apostrophes
    joke = joke.replace("<APOST>", "'")
    
    joke = joke.replace("<ELLIPSIS>", " ... ")
    
    # Remove space around apostrophes (edge cases)
    joke = re.sub(r"\s+'\s+", "'", joke)
    
    return joke.strip()
data["joke_cleaned"] = data.joke.apply(lambda x: clean_jokes(x))
data = data[data.joke_cleaned.notnull()]

In [4]:
print(clean_jokes("What no rEally?!! hahaha... . ..!!!!"))

what no really ? ! hahaha ...  !


In [5]:
data.joke_cleaned.to_list()[:10]

["whoever said imitation is the sincerest form of flattery hasn't had a 7yo mimicking their every word for the last 10 minutes .",
 "you're so fat , they oughta call your dick gary oldman  ... cause it always disappears into a roll .",
 "i couldn't sleep last night . i was wondering where the sun went then it dawned on me",
 "what's 11 2 ? the cowboys",
 'i just got done apologizing to my barbershop quartet i gathered them to sing a song about a bucket with a lot of water in it . it turned out to be solo .',
 'how do you know that a russian oligarch is serious about a deal ? he went to jared .',
 'don\'t expect a " bless you " after the 4th sneeze ... get your shit together .',
 'dj scratches a sick mix crowd goes wild dj scratches a puppy\'s ear crowd " awws " dj scratches lotto ticket crowd " oohs " wins 1',
 'my girlfriend told me she enjoys celebrity impressions in bed , tonight i tried jim carrey apparentley " like a glove " is crossing the line',
 "i was in a cab and the cab driv

In [6]:
data["word_count"] = data.joke_cleaned.apply(lambda x: len(x.split()))

In [7]:
full_text = " ".join(data.joke_cleaned.to_list())

In [8]:
total_words = full_text.split()
unique_words = set(total_words)
print(f"""
total characters  : {len(full_text)}
total words       : {len(total_words)}
total unique words: {len(unique_words)}
""")


total characters  : 108057689
total words       : 23067155
total unique words: 185855



In [9]:
freq = Counter(total_words)

In [10]:
freq.most_common(10)

[('.', 1006799),
 ('the', 951720),
 ('a', 788790),
 (',', 728396),
 ('"', 718161),
 ('to', 449766),
 ('i', 440778),
 ('and', 420683),
 ('?', 405965),
 ('you', 337722)]

In [11]:
data.to_csv(r'..\data\processed\final_cleaned_jokes.csv', index=False)

In [12]:
with open('..\data\processed\jokes_text.txt','w') as f:
    f.write(full_text)

## Tokenizer

In [44]:
with open('..\data\processed\jokes_text.txt','r') as f:
    full_text = f.read()

In [50]:
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel()
# tokenizer.pre_tokenizer = Whitespace()

special_tokens = [
    "[S]",      # Start of sequence
    "[/S]",     # End of sequence
    "[PAD]",    # Padding
    "[UNK]",    # Unknown
    "[MASK]",   
    "[USER]",
    "[JOKE]"
]

trainer = BpeTrainer(
    vocab_size=52000,
    min_frequency=2,
    special_tokens=special_tokens
)
tokenizer.train(files=['..\data\processed\jokes_text.txt'], trainer=trainer)


In [51]:
tokenizer.decoder = ByteLevelDecoder()

tokenizer.enable_padding(pad_id=tokenizer.token_to_id("[PAD]"), pad_token="[PAD]")

tokenizer.enable_truncation(max_length=256,stride=0,strategy='longest_first',direction='right')

tokenizer.post_processor = TemplateProcessing(
    single="[S] $A [/S]",
    pair="[S] $A [JOKE] $B [/S]",
    special_tokens=[
        ("[S]", tokenizer.token_to_id("[S]")), 
        ("[JOKE]", tokenizer.token_to_id("[JOKE]")),
        ("[/S]", tokenizer.token_to_id("[/S]")),
    ],
)

In [52]:
output = tokenizer.encode("tell me a joke about elephants, rain, trunk", "this is a joke about elephants, rain, trunk")
output.tokens

['[S]',
 'Ġtell',
 'Ġme',
 'Ġa',
 'Ġjoke',
 'Ġabout',
 'Ġelephants',
 ',',
 'Ġrain',
 ',',
 'Ġtrunk',
 '[JOKE]',
 'Ġthis',
 'Ġis',
 'Ġa',
 'Ġjoke',
 'Ġabout',
 'Ġelephants',
 ',',
 'Ġrain',
 ',',
 'Ġtrunk',
 '[/S]']

In [53]:
tokenizer.get_vocab_size()

52000

In [54]:
tokenizer.save(r'..\data\processed\tokenizer.json')